# RNN Text Generation with PyTorch

This notebook follows the README workflow:
1. Load and clean corpus
2. Tokenize into words
3. Build token IDs
4. Create input-output pairs
5. Build embedding + RNN model
6. Train and generate text

In [1]:
# Step 0: Imports and basic setup
from pathlib import Path
import re
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility for easier debugging/comparison
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Step 1: Load and Clean Corpus
We read text from `corpus.txt` and apply lightweight cleaning suitable for word-level tokenization.

In [2]:
# Resolve corpus path whether notebook is run from project root or code/
possible_paths = [Path("corpus.txt"), Path("..") / "corpus.txt"]
corpus_path = next((p for p in possible_paths if p.exists()), None)
if corpus_path is None:
    raise FileNotFoundError("Could not find corpus.txt in current or parent directory.")

raw_text = corpus_path.read_text(encoding="utf-8")

# Lowercase + normalize whitespace; keep punctuation as separate tokens later
text = raw_text.lower()
text = re.sub(r"\s+", " ", text).strip()

print(f"Loaded corpus from: {corpus_path}")
print(f"Total characters: {len(text):,}")
print("Preview:", text[:200], "...")

Loaded corpus from: ..\corpus.txt
Total characters: 19,373
Preview: ﻿why did beethoven get rid of his chickens? all they ever said was, bach, bach, bach! what did 20 do when it was hungry? twenty-eight. why is grass so dangerous? because it's full of blades! why are m ...


## Step 2 and Step 3: Tokenize and Build Vocabulary
We use regex to tokenize words and punctuation, then map tokens to integer IDs.

In [3]:
# Word-level tokenization + punctuation tokens
tokens = re.findall(r"[a-z0-9']+|[^\w\s]", text)

# Build vocabulary (sorted for deterministic IDs)
vocab = sorted(set(tokens))
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

token_ids = [stoi[t] for t in tokens]

print(f"Total tokens: {len(tokens):,}")
print(f"Vocabulary size: {len(vocab):,}")
print("First 20 tokens:", tokens[:20])

Total tokens: 4,440
Vocabulary size: 1,223
First 20 tokens: ['\ufeff', 'why', 'did', 'beethoven', 'get', 'rid', 'of', 'his', 'chickens', '?', 'all', 'they', 'ever', 'said', 'was', ',', 'bach', ',', 'bach', ',']


## Step 4: Create Input-Output Training Pairs
Input is a fixed-length sequence of token IDs, and target is the next token ID.

In [4]:
class NextTokenDataset(Dataset):
    """Creates (context_window, next_token) pairs for next-token prediction."""

    def __init__(self, ids, seq_len=8):
        self.ids = ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.ids) - self.seq_len)

    def __getitem__(self, idx):
        x = torch.tensor(self.ids[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.ids[idx + self.seq_len], dtype=torch.long)
        return x, y

SEQ_LEN = 8
BATCH_SIZE = 64

dataset = NextTokenDataset(token_ids, seq_len=SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Training pairs: {len(dataset):,}")
xb, yb = next(iter(loader))
print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)

Training pairs: 4,432
Input batch shape: torch.Size([64, 8])
Target batch shape: torch.Size([64])


## Step 5 and Step 6: Embedding + RNN Model
The model learns token context using an embedding layer and a GRU.

In [5]:
class RNNTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        emb = self.embedding(x)
        out, h = self.rnn(emb, h)
        logits = self.fc(out[:, -1, :])
        return logits, h

VOCAB_SIZE = len(vocab)
model = RNNTextGenerator(VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

RNNTextGenerator(
  (embedding): Embedding(1223, 128)
  (rnn): GRU(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1223, bias=True)
)


## Step 7: Train the Model
For quick experimentation, start with a small number of epochs and increase later.

In [6]:
EPOCHS = 10

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        # Gradient clipping helps stabilize recurrent training
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / max(1, len(loader))
    print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f}")

Epoch 01/10 | Loss: 6.2007
Epoch 02/10 | Loss: 5.3774
Epoch 03/10 | Loss: 4.7682
Epoch 04/10 | Loss: 4.1831
Epoch 05/10 | Loss: 3.6397
Epoch 06/10 | Loss: 3.1203
Epoch 07/10 | Loss: 2.6287
Epoch 08/10 | Loss: 2.1654
Epoch 09/10 | Loss: 1.7431
Epoch 10/10 | Loss: 1.3589


## Step 8 to Step 10: Generate Text from a Seed Prompt
We provide starting words, sample next tokens iteratively, and decode IDs back into text.

In [8]:
def sample_next_token(logits, temperature=1.0):
    # Temperature controls randomness; lower = safer, higher = more creative
    logits = logits / max(temperature, 1e-6)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

def generate_text(model, seed_text, max_new_tokens=40, temperature=0.9):
    model.eval()

    seed_tokens = re.findall(r"[a-z0-9']+|[^\w\s]", seed_text.lower())
    seed_tokens = [t for t in seed_tokens if t in stoi]

    if not seed_tokens:
        raise ValueError("Seed has no known tokens. Try words from the corpus.")

    generated = seed_tokens[:]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = generated[-SEQ_LEN:]
            context_ids = [stoi[t] for t in context]

            # Left-pad so context always has fixed length
            if len(context_ids) < SEQ_LEN:
                context_ids = [context_ids[0]] * (SEQ_LEN - len(context_ids)) + context_ids

            x = torch.tensor([context_ids], dtype=torch.long, device=device)
            logits, _ = model(x)
            next_id = sample_next_token(logits[0], temperature=temperature)
            generated.append(itos[next_id])

    # Simple detokenization for readable output
    out = ' '.join(generated)
    out = re.sub(r"\s+([.,!?;:])", r"\1", out)
    return out

seed = "I am "
print("Seed:", seed)
print("\nGenerated text:")
print(generate_text(model, seed_text=seed, max_new_tokens=50, temperature=0.85))

Seed: I am 

Generated text:
i am my husband down squared to level. beers sam were say to hate this hate me. its raining jamaica closed. you might up them, i went found was mark. which do astronauts like in learn? because dawn old school. what do you call a
